In [9]:
import MetaTrader5 as mt5


# Paramètres de votre compte
ACCOUNT_LOGIN = 10013079704       # Votre numéro de compte (int)
ACCOUNT_PASSWORD = "Es*4KdFw"  # Votre mot de passe (str)
ACCOUNT_SERVER = "MetaQuotes-Demo" # Nom exact du serveur broker (str)

# Initialisation et connexion explicite
if not mt5.initialize(login=ACCOUNT_LOGIN, password=ACCOUNT_PASSWORD, server=ACCOUNT_SERVER):
    print("Erreur de connexion:", mt5.last_error())
    quit()

# Vérification sécurisée des informations du compte
account_info = mt5.account_info()
if account_info is None:
    print("Échec de la récupération des informations du compte. Vérifiez vos identifiants/serveur.")
    mt5.shutdown()
else:
    print(f"MT5 version: {mt5.version()}")
    print(f"Connecté au broker: {account_info.company}")
    print(f"Compte: {account_info.login}")

MT5 version: (500, 6249, '8 Oct 2026')
Connecté au broker: MetaQuotes Ltd.
Compte: 10013079704


In [12]:
import MetaTrader5 as mt5
import pandas as pd
from datetime import datetime

mt5.initialize()

utc_from = datetime(2000, 1, 1)
utc_to   = datetime(2026, 8, 31)

# Télécharger EURUSD en D1 (Daily)
rates = mt5.copy_rates_range(
    "EURUSD",          # Symbole
    mt5.TIMEFRAME_D1,  # Timeframe: D1, H4, H1, M15, M5, M1
    utc_from,
    utc_to
)

df = pd.DataFrame(rates)
df['time'] = pd.to_datetime(df['time'], unit='s')
df.set_index('time', inplace=True)

print(df.shape)   # (504, 7)
print(df.head())  # open, high, low, close, tick_volume...

mt5.shutdown()

(6930, 7)
              open    high     low   close  tick_volume  spread  real_volume
time                                                                        
2000-01-03  1.0073  1.0278  1.0054  1.0246         6642      50            0
2000-01-04  1.0243  1.0340  1.0213  1.0292         7339      50            0
2000-01-05  1.0293  1.0402  1.0284  1.0326         6570      50            0
2000-01-06  1.0325  1.0415  1.0272  1.0330         7223      50            0
2000-01-07  1.0329  1.0332  1.0260  1.0298         5689      50            0


True

In [13]:
import numpy as np
import pandas as pd

np.random.seed(42)
n      = 504          # 2 ans × 252 jours de trading
mu     = 0.0001       # drift journalier
sigma  = 0.004        # volatilité journalière EURUSD
S0     = 1.0700       # prix de départ

# Geometric Brownian Motion
returns     = np.random.normal(mu, sigma, n)
close_arr   = S0 * np.exp(np.cumsum(returns))
daily_range = np.random.uniform(0.002, 0.008, n)

open_arr    = np.roll(close_arr, 1); open_arr[0] = S0
high_arr    = np.maximum(open_arr, close_arr) + daily_range * 0.5
low_arr     = np.minimum(open_arr, close_arr) - daily_range * 0.5

dates = pd.date_range('2023-01-02', periods=n, freq='B')
df = pd.DataFrame({
    'Open': np.round(open_arr, 5),  'High': np.round(high_arr, 5),
    'Low':  np.round(low_arr, 5),   'Close': np.round(close_arr, 5),
    'Volume': np.random.randint(50000, 200000, n)
}, index=dates)

print(df.shape)   # (504, 5)

(504, 5)


In [14]:
# Exploration systématique
print(df.shape)        # (504, 5)
print(df.info())       # Types, nulls
print(df.describe())   # Stats

# Stats clés EURUSD
print(f"Période:        {df.index[0].date()} → {df.index[-1].date()}")
print(f"Prix moyen:     {df['Close'].mean():.5f}")
print(f"Prix min:       {df['Close'].min():.5f}")
print(f"Prix max:       {df['Close'].max():.5f}")
print(f"Volatilité:     {df['Close'].pct_change().std():.4%}")
print(f"Valeurs nulles: {df.isnull().sum().sum()}")

(504, 5)
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 504 entries, 2023-01-02 to 2024-12-05
Freq: B
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Open    504 non-null    float64
 1   High    504 non-null    float64
 2   Low     504 non-null    float64
 3   Close   504 non-null    float64
 4   Volume  504 non-null    int32  
dtypes: float64(4), int32(1)
memory usage: 21.7 KB
None
             Open        High         Low       Close         Volume
count  504.000000  504.000000  504.000000  504.000000     504.000000
mean     1.092320    1.096617    1.088182    1.092479  125384.551587
std      0.044309    0.044529    0.044106    0.044372   42857.353769
min      1.024340    1.026930    1.020880    1.024340   50207.000000
25%      1.048440    1.052265    1.044033    1.048440   86255.500000
50%      1.089435    1.093805    1.086255    1.089535  126042.500000
75%      1.134078    1.139047    1.129587    1.134672  162357.

In [15]:
# Exploration systématique
print(df.shape)        # (504, 5)
print(df.info())       # Types, nulls
print(df.describe())   # Stats

# Stats clés EURUSD
print(f"Période:        {df.index[0].date()} → {df.index[-1].date()}")
print(f"Prix moyen:     {df['Close'].mean():.5f}")
print(f"Prix min:       {df['Close'].min():.5f}")
print(f"Prix max:       {df['Close'].max():.5f}")
print(f"Volatilité:     {df['Close'].pct_change().std():.4%}")
print(f"Valeurs nulles: {df.isnull().sum().sum()}")#Moving Averages 
df['SMA_5']  = df['Close'].rolling(5).mean()   # Tendance 1 semaine
df['SMA_20'] = df['Close'].rolling(20).mean()  # Tendance 1 mois
df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()
df['EMA_26'] = df['Close'].ewm(span=26, adjust=False).mean()

#MACD
df['MACD'] = df['EMA_12'] - df['EMA_26']       # Momentum

#Bollinger Bands
roll = df['Close'].rolling(20)
df['BB_upper'] = roll.mean() + 2 * roll.std()
df['BB_lower'] = roll.mean() - 2 * roll.std()
df['BB_width'] = df['BB_upper'] - df['BB_lower']  # Volatilité

# Price Lags (mémoire du marché)
df['Close_Lag1'] = df['Close'].shift(1)  # Close d'hier
df['Close_Lag2'] = df['Close'].shift(2)  # Close avant-hier

# Features de bougie
df['High_Low_Range']  = df['High'] - df['Low']
df['Open_Close_Diff'] = df['Close'] - df['Open']
df['Volatility_5']    = df['Close'].pct_change().rolling(5).std()

(504, 5)
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 504 entries, 2023-01-02 to 2024-12-05
Freq: B
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Open    504 non-null    float64
 1   High    504 non-null    float64
 2   Low     504 non-null    float64
 3   Close   504 non-null    float64
 4   Volume  504 non-null    int32  
dtypes: float64(4), int32(1)
memory usage: 21.7 KB
None
             Open        High         Low       Close         Volume
count  504.000000  504.000000  504.000000  504.000000     504.000000
mean     1.092320    1.096617    1.088182    1.092479  125384.551587
std      0.044309    0.044529    0.044106    0.044372   42857.353769
min      1.024340    1.026930    1.020880    1.024340   50207.000000
25%      1.048440    1.052265    1.044033    1.048440   86255.500000
50%      1.089435    1.093805    1.086255    1.089535  126042.500000
75%      1.134078    1.139047    1.129587    1.134672  162357.

In [16]:
# Target = Close du jour suivant (shift(-1))
df['Target'] = df['Close'].shift(-1)

# Supprimer les NaN créés par les rolling windows et le shift
df_clean = df.dropna()

print(f"Shape avant dropna: {df.shape}")     # (504, 22)
print(f"Shape après  dropna: {df_clean.shape}") # (454, 22)
print(f"Lignes perdues: {len(df) - len(df_clean)}")  # 50

# Vérification: la target décalée d'une ligne
print(df_clean[['Close', 'Target']].head(4))

Shape avant dropna: (504, 19)
Shape après  dropna: (484, 19)
Lignes perdues: 20
              Close   Target
2023-01-27  1.05755  1.06387
2023-01-30  1.06387  1.06302
2023-01-31  1.06302  1.06341
2023-02-01  1.06341  1.05748


In [17]:
features = ['Open', 'High', 'Low', 'Close', 'SMA_5', 'SMA_20',
            'EMA_12', 'MACD', 'BB_width', 'Close_Lag1', 'Close_Lag2',
            'High_Low_Range', 'Open_Close_Diff', 'Volatility_5']

X = df_clean[features].values
y = df_clean['Target'].values

# Split TEMPOREL: les 80% premiers pour train, les 20% pour test
split_idx = int(len(X) * 0.80)          # index 363
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Train: {len(X_train)} samples")  # 363
print(f"Test:  {len(X_test)} samples")   # 91

Train: 387 samples
Test:  97 samples


In [18]:
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()
scaler_y = StandardScaler()

# fit_transform sur TRAIN seulement!
X_train_s = scaler_X.fit_transform(X_train)
y_train_s = scaler_y.fit_transform(
                y_train.reshape(-1, 1)).ravel()

# transform seulement sur TEST (pas de fit!)
X_test_s  = scaler_X.transform(X_test)
y_test_s  = scaler_y.transform(
                y_test.reshape(-1, 1)).ravel()

print(f"X train mean ≈ 0:  {X_train_s.mean(axis=0)[:3].round(4)}")
print(f"X test  mean:      {X_test_s.mean(axis=0)[:3].round(4)}")

X train mean ≈ 0:  [ 0. -0.  0.]
X test  mean:      [1.9935 1.9974 1.9785]


In [19]:
from sklearn.linear_model import LinearRegression

# Créer et entraîner le modèle
model = LinearRegression()
model.fit(X_train_s, y_train_s)

# Inspecter le modèle entraîné
print(f"Intercept (β₀):       {model.intercept_:.6f}")
print(f"R² (train):           {model.score(X_train_s, y_train_s):.6f}")
print(f"Nb de coefficients:   {len(model.coef_)}")

# Voir les coefficients par feature
coef_df = pd.DataFrame({
    'Feature':     features,
    'Coefficient': model.coef_
}).sort_values('Coefficient', ascending=False)

print(coef_df.head(5))

Intercept (β₀):       -0.000000
R² (train):           0.988027
Nb de coefficients:   14
           Feature  Coefficient
2              Low    30.785083
1             High    30.628784
6           EMA_12     0.201523
4            SMA_5     0.087398
11  High_Low_Range     0.013496


In [20]:
y_pred_s = model.predict(X_test_s)

# Dénormaliser pour retrouver les vrais prix EURUSD
y_pred = scaler_y.inverse_transform(
            y_pred_s.reshape(-1, 1)).ravel()

# Comparer prédictions vs réalité
results = pd.DataFrame({
    'Date':     df_clean.index[split_idx:],
    'Actual':   y_test,
    'Predicted': y_pred,
    'Error':    y_test - y_pred
})

results['Error_pips'] = results['Error'] * 10000  # Pips

print(results.head(6))

        Date   Actual  Predicted     Error  Error_pips
0 2024-07-23  1.14090   1.142500 -0.001600  -16.003180
1 2024-07-24  1.14157   1.141509  0.000061    0.605685
2 2024-07-25  1.14403   1.142291  0.001739   17.390052
3 2024-07-26  1.14741   1.144526  0.002884   28.837090
4 2024-07-29  1.14237   1.147633 -0.005263  -52.627419
5 2024-07-30  1.13550   1.143070 -0.007570  -75.704471


In [21]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

# Direction Accuracy (montée ou descente?)
actual_dir    = np.sign(np.diff(y_test))
predicted_dir = np.sign(np.diff(y_pred))
dir_accuracy  = (actual_dir == predicted_dir).mean()

print(f"MSE:               {mse:.8f}")
print(f"RMSE:              {rmse:.6f}  ({rmse*10000:.1f} pips)")
print(f"MAE:               {mae:.6f}  ({mae*10000:.1f} pips)")
print(f"R²:                {r2:.4f}")
print(f"Direction Accuracy: {dir_accuracy:.2%}")

MSE:               0.00002532
RMSE:              0.005032  (50.3 pips)
MAE:               0.004134  (41.3 pips)
R²:                0.8806
Direction Accuracy: 55.21%


In [23]:
# Récupérer la dernière ligne de données connues
last_row    = df_clean[features].iloc[-1].values.reshape(1, -1)
last_date   = df_clean.index[-1]
last_close  = df_clean['Close'].iloc[-1]

# Normaliser avec le scaler déjà entraîné
last_scaled = scaler_X.transform(last_row)

# Prédire
next_close_s = model.predict(last_scaled)
next_close   = scaler_y.inverse_transform(
                   next_close_s.reshape(-1, 1))[0][0]

# Résultat en pips
diff_pips = (next_close - last_close) * 10000
direction = "HAUSSE" if diff_pips > 0 else "BAISSE"

print(f"Dernier Close:     {last_close:.5f}  ({last_date.date()})")
print(f"Prochain Close:    {next_close:.5f}")
print(f"Mouvement prédit:  {diff_pips:+.1f} pips")
print(f"Direction:         {direction}")

Dernier Close:     1.14727  (2024-12-04)
Prochain Close:    1.14752
Mouvement prédit:  +2.5 pips
Direction:         HAUSSE


In [24]:
import numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

#Données (MT5 ou simulées)
np.random.seed(42); n = 504; S0 = 1.07
close = S0 * np.exp(np.cumsum(np.random.normal(0.0001, 0.004, n)))
df = pd.DataFrame({'Open': np.roll(close,1), 'Close': close,
    'High': close + 0.003, 'Low': close - 0.003},
    index=pd.date_range('2023-01-02', periods=n, freq='B'))
df.iloc[0, 0] = S0

#Features
df['SMA_5'] = df['Close'].rolling(5).mean()
df['SMA_20']= df['Close'].rolling(20).mean()
df['EMA_12']= df['Close'].ewm(span=12).mean()
df['MACD']  = df['Close'].ewm(span=12).mean() - df['Close'].ewm(span=26).mean()
df['Lag1']  = df['Close'].shift(1)
df['Lag2']  = df['Close'].shift(2)
df['Target']= df['Close'].shift(-1)
df = df.dropna()

#Split & Scale
feats = ['Open','Close','SMA_5','SMA_20','EMA_12','MACD','Lag1','Lag2']
X, y  = df[feats].values, df['Target'].values
idx   = int(len(X) * .8)
Xtr, Xte, ytr, yte = X[:idx], X[idx:], y[:idx], y[idx:]
sx, sy = StandardScaler(), StandardScaler()
Xtr_s  = sx.fit_transform(Xtr);  ytr_s = sy.fit_transform(ytr.reshape(-1,1)).ravel()
Xte_s  = sx.transform(Xte);      yte_s = sy.transform(yte.reshape(-1,1)).ravel()

#Modèle
model  = LinearRegression().fit(Xtr_s, ytr_s)
ypred  = sy.inverse_transform(model.predict(Xte_s).reshape(-1,1)).ravel()

# Métriques
print(f"RMSE: {np.sqrt(mean_squared_error(yte, ypred))*10000:.1f} pips")
print(f"R²:   {r2_score(yte, ypred):.4f}")

RMSE: 50.5 pips
R²:   0.8797


In [25]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline

# Pipeline: scaler + modèle
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LinearRegression())
])

# TimeSeriesSplit: 5 folds, SANS data leakage
tscv = TimeSeriesSplit(n_splits=5)

# Évaluer avec R²
scores = cross_val_score(pipe, X, y,
                         cv=tscv,
                         scoring='r2')

print(f"R² par fold: {scores.round(3)}")
print(f"R² moyen:    {scores.mean():.3f}")
print(f"R² std:      {scores.std():.3f}")

R² par fold: [0.841 0.878 0.966 0.911 0.881]
R² moyen:    0.895
R² std:      0.042


In [26]:
# Linear Regression (Ordinary Least Squares)
from sklearn.linear_model import LinearRegression
model = LinearRegression()

# Ridge (L2)
from sklearn.linear_model import Ridge
model = Ridge(alpha=1.0)

# Lasso (L1)
from sklearn.linear_model import Lasso
model = Lasso(alpha=0.01)